In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Project directories
PROJECT_DIR = Path(
    '/content/drive/MyDrive/INF8250AE_research_project'
)

RESULTS_DIR = PROJECT_DIR / 'results' / 'feasibility'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

import subprocess

DREAMER_DIR = Path('/content/dreamerv3')

if not DREAMER_DIR.exists():
    subprocess.run(
        [
            'git', 'clone',
            'https://github.com/danijar/dreamerv3.git',
            str(DREAMER_DIR)
        ],
        check=True
    )

%cd /content/dreamerv3

!python -m pip install -r requirements.txt
!python -m pip install dm_control

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/INF8250AE_research_project
/content/dreamerv3


In [3]:
import jax
import platform

print("Python:", platform.python_version())
print("JAX:", jax.__version__)
print("Available devices:", jax.devices())

Python: 3.12.13
JAX: 0.4.33
Available devices: [CudaDevice(id=0)]


In [4]:
# Experimental configuration

ENVIRONMENT = "dmc_cartpole_swingup"

RECONSTRUCTION_WEIGHT = 1.0
SEED = 42

TRAINING_STEPS = 10_000
TRAIN_RATIO = 32

# Local directory for faster training
RUN_NAME = "baseline_w1_k0_seed42"

LOCAL_LOG_DIR = Path("/content/dreamerv3_runs") / RUN_NAME
DRIVE_LOG_DIR = RESULTS_DIR / RUN_NAME

print("Environment:", ENVIRONMENT)
print("Reconstruction weight:", RECONSTRUCTION_WEIGHT)
print("Training steps:", TRAINING_STEPS)
print("Training ratio:", TRAIN_RATIO)

Environment: dmc_cartpole_swingup
Reconstruction weight: 1.0
Training steps: 10000
Training ratio: 32


In [5]:
# Install MuJoCo software rendering dependencies

!apt-get update -qq
!apt-get install -y libosmesa6 libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libosmesa6 is already the newest version (23.2.1-1ubuntu3.1~22.04.4).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 265 not upgraded.


In [6]:
import os
import sys
from pathlib import Path

# Headless rendering for Google Colab
env_vars = os.environ.copy()
env_vars["MUJOCO_GL"] = "osmesa"
env_vars["PYOPENGL_PLATFORM"] = "osmesa"

# Optional: enable detailed JAX compilation diagnostics
env_vars["JAX_LOG_COMPILES"] = "1"

LOCAL_LOG_DIR = Path(LOCAL_LOG_DIR)
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, "-u", "dreamerv3/main.py",
    "--configs", "dmc_proprio",
    "--task", ENVIRONMENT,
    "--env.dmc.image", "False",
    "--agent.loss_scales.rec", str(RECONSTRUCTION_WEIGHT),
    "--seed", str(SEED),
    "--run.steps", str(TRAINING_STEPS),
    "--run.train_ratio", str(TRAIN_RATIO),
    "--logdir", str(LOCAL_LOG_DIR),
]

print("Command prepared.")
print("Log directory:", LOCAL_LOG_DIR)

Command prepared.
Log directory: /content/dreamerv3_runs/baseline_w1_k0_seed42


In [12]:
# Lightweight feasibility test

import time

TEST_LOG_DIR = "/content/dreamerv3_runs/feasibility_light_v1"

start_time = time.perf_counter()

!MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa \
python -u dreamerv3/main.py \
    --configs dmc_proprio \
    --task dmc_cartpole_swingup \
    --env.dmc.image False \
    --agent.loss_scales.rec 1.0 \
    --seed 42 \
    --run.envs 1 \
    --batch_size 4 \
    --batch_length 32 \
    --run.train_ratio 8 \
    --run.steps 1000 \
    --logdir {TEST_LOG_DIR}

elapsed = time.perf_counter() - start_time

print(f"\nTotal execution time: {elapsed / 60:.2f} minutes")

---  ___                           __   ______ ---
--- |   \ _ _ ___ __ _ _ __  ___ _ \ \ / /__ / ---
--- | |) | '_/ -_) _` | '  \/ -_) '/\ V / |_ \ ---
--- |___/|_| \___\__,_|_|_|_\___|_|  \_/ |___/ ---
Replica: 0 / 1
Logdir: /content/dreamerv3_runs/feasibility_light_v1
Run script: train
Observations
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
  position         Space(float32, shape=(3,), low=-inf, high=inf)
  velocity         Space(float32, shape=(2,), low=-inf, high=inf)
Actions
  action           Space(float32, shape=(1,), low=-1.0, high=1.0)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(512,), low=-inf, high=inf)
  dyn/stoch        Space(

In [17]:
# Diagnostic: environment interaction without training updates

import time

TEST_LOG_DIR = "/content/dreamerv3_runs/interaction_diagnostic_new"

start_time = time.perf_counter()

!MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa \
python -u dreamerv3/main.py \
    --configs dmc_proprio \
    --task dmc_cartpole_swingup \
    --env.dmc.image False \
    --jax.platform cpu \
    --run.envs 1 \
    --batch_size 4 \
    --batch_length 32 \
    --run.train_ratio 0 \
    --run.steps 100 \
    --run.log_every 5 \
    --run.report_every 1000 \
    --logdir {TEST_LOG_DIR}

elapsed = time.perf_counter() - start_time

print(f"\nTotal execution time: {elapsed / 60:.2f} minutes")

---  ___                           __   ______ ---
--- |   \ _ _ ___ __ _ _ __  ___ _ \ \ / /__ / ---
--- | |) | '_/ -_) _` | '  \/ -_) '/\ V / |_ \ ---
--- |___/|_| \___\__,_|_|_|_\___|_|  \_/ |___/ ---
Replica: 0 / 1
Logdir: /content/dreamerv3_runs/interaction_diagnostic_new
Run script: train
Observations
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
  position         Space(float32, shape=(3,), low=-inf, high=inf)
  velocity         Space(float32, shape=(2,), low=-inf, high=inf)
Actions
  action           Space(float32, shape=(1,), low=-1.0, high=1.0)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(512,), low=-inf, high=inf)
  dyn/stoch        